<a href="https://colab.research.google.com/github/Kwakjaemin1007/Like-lion_-Smart-farm/blob/main/smartfarm_strawberry_aihub_2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path

In [ ]:
str_data = "01.원천데이터"
str_label = "02.라벨링데이터"
str_common = "01.딸기_001.설향"
str_normal = "01.정상"
str_rot = "02.역병"
str_wilt = "03.시들음병"
str_burn = "04.잎끝마름"
str_yellowing = "05.황화"

base_path = Path("/content/drive/MyDrive/bootcamp_proj/proj1/AI_HUB_strawberry/01-1.정식개방데이터/Validation")
data_path = base_path / str_data / str_common
label_path = base_path / str_label / str_common
disease_categories = [str_normal, str_rot, str_wilt, str_burn, str_yellowing]

In [ ]:
from zipfile import ZipFile

In [ ]:
cat_name_to_id = {
    '잎': 0,
    '줄기': 1,
    "화방": 2,
    "과실": 3
}

In [ ]:
part_cat = {
    0: 'leaf',
    1: 'stem',
    2: "flower",
    3: "fruit",
}

In [ ]:
disease_cat = {
    0: "normal",
    1: "rot",
    2: "wilt",
    3: "burn",
    4: "yellowing"
}

In [ ]:
output_detection_path = Path("/content/yolo_strawberry/")
output_classification_path = Path("/content/illness_detection/")

In [ ]:
from shutil import copy
from tqdm.notebook import tqdm
from tqdm.contrib import tzip
from zipfile import ZipFile, ZipInfo
import zipfile
from tempfile import TemporaryDirectory
import json
import albumentations as A
import numpy as np
import cv2

In [ ]:
train_split, test_split, valid_split = 0.8, 0.1, 0.1

# Zip the file if not exist

In [ ]:
output_detection_path.mkdir(exist_ok=True)
output_train_img_path = output_detection_path / "images/train"
output_train_label_path = output_detection_path / "labels/train"
output_test_img_path = output_detection_path / "images/test"
output_test_label_path = output_detection_path / "labels/test"
output_val_img_path = output_detection_path / "images/val"
output_val_label_path = output_detection_path / "labels/val"
output_train_img_path.mkdir(parents=True, exist_ok=True)
output_train_label_path.mkdir(parents=True, exist_ok=True)
output_test_img_path.mkdir(parents=True, exist_ok=True)
output_test_label_path.mkdir(parents=True, exist_ok=True)
output_val_img_path.mkdir(parents=True, exist_ok=True)
output_val_label_path.mkdir(parents=True, exist_ok=True)

output_classification_path.mkdir(parents=True, exist_ok=True)
for next_disease in disease_cat.values():
    (output_classification_path / next_disease).mkdir(parents=True, exist_ok=True)

In [ ]:
output_img_paths = {
    "train": output_train_img_path,
    "test": output_test_img_path,
    "val": output_val_img_path,
}
output_label_paths = {
    "train": output_train_label_path,
    "test": output_test_label_path,
    "val": output_val_label_path,
}

Pipeline

Image -> YOLO -> Leaves / Stems

Leaves / Stems -> Resnet (or sth) -> Illness

(+Ensenble?)

# Preprocessing if zip file not exists

In [ ]:
from collections import Counter
IMGSZ = 640

transform = A.Compose([
    A.LongestMaxSize(max_size=IMGSZ),
], bbox_params=A.BboxParams(format="coco", label_fields=['class_ids']))

disease_counter = Counter()
for disease_idx, next_disease in enumerate(tqdm(disease_categories)):
    # Get zip file path
    cat_data_zip_path = base_path / str_data / f"VS_{str_common}_{next_disease}.zip"
    cat_label_zip_path = base_path / str_label / f"VL_{str_common}_{next_disease}.zip"


    with ZipFile(cat_data_zip_path) as cat_data_zip, ZipFile(cat_label_zip_path) as cat_label_zip:
        img_list = sorted(cat_data_zip.infolist(), key=lambda x: x.filename)
        label_list = sorted(cat_label_zip.infolist(), key=lambda x: x.filename)

        assert len(img_list) == len(label_list)
        data_cnt = len(img_list)

        # Using tzip to zip image an annotation - using assertion to make sure those are pair
        for data_idx, (next_img_zipinfo, next_label_zipinfo) in enumerate(tzip(img_list, label_list)):

            # Get current split
            current_progress = data_idx / data_cnt
            current_split = ""
            if current_progress <= train_split:
                current_split = "train"
            elif current_progress <= train_split + test_split:
                current_split = "test"
            else:
                current_split = "val"

            # Select output path according to current split
            output_img_path = output_img_paths[current_split]
            output_label_path = output_label_paths[current_split]

            next_img_zipinfo: ZipInfo
            next_label_zipinfo: ZipInfo
            assert Path(next_img_zipinfo.filename).stem == Path(next_label_zipinfo.filename).stem
            img_name = Path(next_img_zipinfo.filename).name
            label_name = Path(next_label_zipinfo.filename).with_suffix(".txt").name


            label_data = json.loads(cat_label_zip.read(next_label_zipinfo))

            # Load image
            cat_data_zip.extract(next_img_zipinfo, output_img_path)
            img_bytes = cat_data_zip.read(next_img_zipinfo)
            img_np = np.frombuffer(img_bytes, dtype=np.int8)
            img = cv2.imdecode(img_np, cv2.IMREAD_COLOR)
            assert img is not None

            law_bboxs = []
            law_labels = []

            # For every annotations in the json file
            for next_annot in label_data['annotations']:
                bbox = list(map(float, next_annot['bbox']))
                law_bboxs.append(bbox)


                # Dataset #1. Leave/stem object detection
                annot_cat_id = int(next_annot['category_id'])
                part_cat_name = label_data['categories'][annot_cat_id]['name']
                if part_cat_name not in ['잎', '줄기']:
                    continue
                part_cat_id = cat_name_to_id[part_cat_name]
                law_labels.append(part_cat_id)

                # Dataset #2. Partial image with disease class
                x, y, w, h = map(int, bbox)
                if w > 0 and h > 0:
                    part_img = img[y: y + h, x : x + w]
                    has_disease = (next_annot['disease_status'] != "N")
                    cur_class = disease_idx if has_disease else 0 # normal
                    disease_counter[cur_class] += 1
                    # Will save this without augmentation
                    cv2.imwrite(f"{output_classification_path / str(disease_cat[cur_class]) / f'{disease_counter[cur_class]:05d}'}.jpg",part_img)



            # Resize using albumentation
            try:
                transformed = transform(image=img, bboxes=law_bboxs, class_ids=law_labels)
                trans_img = transformed['image']
                trans_bboxes = transformed['bboxes']
                trans_class = transformed['class_ids']
            except Exception:
                continue

            # assert trans_img.shape == (640, 640, 3)

            label_file_name = output_label_path / label_name

            h_img = trans_img.shape[0]
            w_img = trans_img.shape[1]

            with open(label_file_name, "w") as f:
                if len(trans_class) != len(trans_bboxes) or len(trans_bboxes) == 0:
                    print(label_file_name)
                    continue
                for next_label, next_bbox in zip(trans_class, trans_bboxes):
                    x_min, y_min, w, h = next_bbox

                    clip = lambda x: max(0, min(1, x))
                    x = clip((x_min + w / 2) / w_img)
                    y = clip((y_min + h / 2) / h_img)
                    w_yolo = clip(w / w_img)
                    h_yolo = clip(h / h_img)

                    f.write(f"{next_label} {x} {y} {w_yolo} {h_yolo}\n")
            if label_file_name.exists():
                cv2.imwrite(str(output_img_path / img_name), trans_img)

print(disease_counter)


  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/7429 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Zip it

In [ ]:
import shutil
shutil.make_archive("strawberry_data_detection", "zip", output_detection_path)
print("detection compressed")
shutil.make_archive("strawberry_data_classification", "zip", output_classification_path)
print("classification compressed")

'/content/strawberry_data-640x640.zip'

In [ ]:
!cp /content/strawberry_data_detection.zip /path/to/drive/strawberry_data_detection.zip
!cp /content/strawberry_data_classification.zip /path/to/drive/strawberry_data_classification.zip

# Else

In [ ]:
data_zip_path = Path("/content/drive/MyDrive/bootcamp_proj/proj1/strawberry_data_640x640.zip")

In [ ]:
!cp {data_zip_path} /content/

In [ ]:
!unzip -q "./strawberry_data_640x640.zip" -d /content/yolo_strawberry